# Fako Online - B-Roll Generation Server

**Wan 2.1 (1.3B) Video Generation**

Generates short B-Roll video clips from text prompts.

### Instructions
1. Set runtime to **T4 GPU**
2. Run all cells in order
3. Copy the ngrok URL to your `.env` file

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
print('Google Drive mounted!')

In [ ]:
# Cell 2: Load Wan 2.1 model from Drive
# Account A models
account_a_dir = '/content/drive/MyDrive/AI_Models'

# Check if model exists
wan_path = f'{account_a_dir}/Wan2.1-1.3B'
if os.path.exists(wan_path):
    print(f'Wan 2.1 model found at {wan_path}')
else:
    print(f'WARNING: Model not found at {wan_path}')
    print('Please run the first-time setup to download the model.')

In [ ]:
# Cell 3: Install dependencies
!pip install -q diffusers transformers accelerate
!pip install -q fastapi uvicorn pyngrok python-multipart
!pip install -q imageio[ffmpeg]
print('Dependencies installed!')

In [ ]:
# Cell 4: Import libraries
import torch
import numpy as np
from diffusers import WanPipeline
from diffusers.utils import export_to_video
import io
import base64

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
# Cell 5: Load Wan 2.1 pipeline
# Load from the downloaded checkpoint
pipe = WanPipeline.from_pretrained(
    f'{account_a_dir}/Wan2.1-1.3B',
    torch_dtype=torch.float16
)
pipe.to('cuda')

print('Wan 2.1 loaded!')

In [ ]:
# Cell 6: Define B-Roll generation function
def generate_broll(prompt, duration=5, num_frames=81):
    """Generate B-Roll video from text prompt."""
    video_frames = pipe(
        prompt=prompt,
        num_frames=num_frames,
        num_inference_steps=20,
        guidance_scale=7.5,
        height=480,
        width=832
    ).frames[0]
    
    output_path = '/content/outputs/broll_clip.mp4'
    export_to_video(video_frames, output_path, fps=24)
    return output_path

print('B-Roll generation function defined!')

In [ ]:
# Cell 7: Create FastAPI server
from fastapi import FastAPI, Form
from fastapi.responses import FileResponse, JSONResponse
import uvicorn

app = FastAPI(title='Fako Online - B-Roll API')

@app.get('/health')
async def health():
    return {'status': 'ok', 'model': 'wan-2.1-1.3b'}

@app.post('/generate-broll')
async def api_generate_broll(text_prompt: str = Form(...), duration: int = Form(5)):
    try:
        video_path = generate_broll(text_prompt, duration)
        return FileResponse(video_path, media_type='video/mp4')
    except Exception as e:
        return JSONResponse({'error': str(e)}, status_code=500)

print('FastAPI server defined!')

In [ ]:
# Cell 8: Start server with ngrok
from pyngrok import ngrok

ngrok.kill()
public_url = ngrok.bind(8001)
print(f'\nPublic URL: {public_url}')
print(f'\nUpdate your .env file:')
print(f'COLAB_BROLL_URL={public_url}')

uvicorn.run(app, host='0.0.0.0', port=8001)